# Interactive Base vs SFT vs DPO — Streamlit demo
Pick one of the 10 fixed test questions, hit **Get answers**, and see all
three models' answers side by side — the winning one is badged, with the
actual reason it won (and, implicitly, why the other two didn't) pulled
straight from your own verified `reports/final_evaluation.md`.

CPU-only, no model loading — this just reads and nicely presents a report
you already built and judged by hand. Not a required deliverable, a demo aid.

`Runtime -> CPU` is fine.

In [138]:
# mount Drive and locate the project
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Domain-SupportSpecialist'

if not os.path.exists(f'{PROJECT}/reports/final_evaluation.md'):
    raise FileNotFoundError(
        "reports/final_evaluation.md is missing -- run notebooks/dpo_alignment.ipynb "
        "(Stage 3) all the way through first, this demo needs that report."
    )
print('Found report at:', f'{PROJECT}/reports/final_evaluation.md')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found report at: /content/drive/MyDrive/Domain-SupportSpecialist/reports/final_evaluation.md


In [139]:
!pip install -q streamlit
!which streamlit

/usr/local/bin/streamlit


## Write the app
The Streamlit app's full source lives in this cell -- `%%writefile` saves it
to `app.py` in the Colab runtime's local disk (not Drive), where
`streamlit run` can execute it.

In [140]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="Base vs SFT vs DPO", page_icon=":material/quiz:", layout="wide")

PROJECT = "/content/drive/MyDrive/Domain-SupportSpecialist"

# the same 10 fixed questions used across every notebook in this project --
# used as anchors so the parser below doesn't get confused by answers that
# span multiple physical lines in the markdown file.
QUESTIONS = [
    "How can I cancel my order after it has been placed?",
    "My package says delivered but I never received it, what do I do?",
    "How long does a refund take to appear on my card?",
    "Can I change the delivery address after checkout?",
    "The item I received is damaged, what are my options?",
    "How do I track my order status?",
    "I was charged twice for one order, how do I fix this?",
    "Can I get a replacement instead of a refund?",
    "What happens if I miss the delivery attempt?",
    "How do I apply a discount code after placing an order?",
]


@st.cache_data
def load_final_evaluation():
    path = f"{PROJECT}/reports/final_evaluation.md"
    with open(path, encoding="utf-8") as f:
        text = f.read()

    # find where each question's row starts -- robust to multi-line cell
    # content (numbered steps etc.), unlike a naive line-by-line parser
    anchors = []
    for q in QUESTIONS:
        marker = f"| {q} |"
        idx = text.find(marker)
        if idx != -1:
            anchors.append((idx, q))
    anchors.sort()

    rows = []
    for i, (idx, q) in enumerate(anchors):
        end = anchors[i + 1][0] if i + 1 < len(anchors) else len(text)
        block = text[idx:end].strip()
        # drop a leading/trailing empty piece from the outer "|"
        parts = [p.strip() for p in block.split("|")]
        parts = [p for p in parts if p != ""] if parts and parts[0] == "" else parts
        # parts: [Question, Base, SFT, DPO, Best Answer, Reason]
        if len(parts) >= 6:
            rows.append({
                "Question": parts[0], "Base": parts[1], "SFT": parts[2],
                "DPO": parts[3], "Best Answer": parts[4], "Reason": parts[5],
            })
    return rows


rows = load_final_evaluation()
by_question = {r["Question"]: r for r in rows}

st.title("Customer Support Assistant")
st.caption(
    f"{len(rows)} fixed test questions loaded, three models, one honest verdict each. "
    "Answers shown here are pre-computed from your own verified evaluation report -- "
    "this demo doesn't run the models live."
)

# ---- session state: a LIVE tally that grows as questions get revealed this session ----
if "asked" not in st.session_state:
    st.session_state.asked = set()
if "tally" not in st.session_state:
    st.session_state.tally = {"Base": 0, "SFT": 0, "DPO": 0}
if "last_question" not in st.session_state:
    st.session_state.last_question = None

# ---- KPI row: live, updates as you ask more questions ----
with st.container(horizontal=True):
    st.metric("Base wins", st.session_state.tally["Base"], border=True)
    st.metric("SFT wins", st.session_state.tally["SFT"], border=True)
    st.metric("DPO wins", st.session_state.tally["DPO"], border=True)
    st.metric("Questions asked", f"{len(st.session_state.asked)} / {len(rows)}", border=True)

# ---- donut + bar chart, both reflect the same live tally ----
# drawn with matplotlib (st.pyplot) instead of st.vega_lite_chart / st.bar_chart --
# those two kept either silently failing to repaint on rerun (a known Vega-Lite/
# Streamlit patch-in-place quirk) or breaking on a `key` argument the installed
# Streamlit version's bar_chart doesn't support yet. A fresh matplotlib figure is
# rebuilt from scratch every single rerun, so there's nothing to go stale.
MODEL_ORDER = ["Base", "SFT", "DPO"]
MODEL_COLORS = ["#94a3b8", "#38bdf8", "#0f766e"]
wins = [st.session_state.tally[m] for m in MODEL_ORDER]

chart_col1, chart_col2 = st.columns(2)
with chart_col1:
    with st.container(border=True):
        st.markdown("**Win share**")
        if sum(wins) > 0:
            fig, ax = plt.subplots(figsize=(2.6, 2.6), dpi=150)
            nonzero = [(m, w, c) for m, w, c in zip(MODEL_ORDER, wins, MODEL_COLORS) if w > 0]
            ax.pie(
                [w for _, w, _ in nonzero],
                labels=[m for m, _, _ in nonzero],
                colors=[c for _, _, c in nonzero],
                autopct=lambda pct: f"{round(pct * sum(wins) / 100)}",
                wedgeprops={"width": 0.45},
                startangle=90,
                textprops={"fontsize": 8},
            )
            ax.set_aspect("equal")
            fig.tight_layout()
            # use_container_width=False -- otherwise Streamlit re-stretches this
            # small figure back out to fill the full column width regardless of figsize
            st.pyplot(fig, clear_figure=True, use_container_width=False)
        else:
            st.caption("Ask a question below to start filling this in.")
with chart_col2:
    with st.container(border=True):
        st.markdown("**Win count**")
        fig, ax = plt.subplots(figsize=(2.6, 2.6), dpi=150)
        ax.bar(MODEL_ORDER, wins, color=MODEL_COLORS)
        ax.set_ylabel("Wins", fontsize=8)
        ax.set_ylim(0, max(wins + [1]) + 1)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.tick_params(labelsize=8)
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)
        fig.tight_layout()
        st.pyplot(fig, clear_figure=True, use_container_width=False)

st.divider()

# ---- Question picker ----
question = st.selectbox("Pick a test question", options=list(by_question.keys()))
show = st.button("Get answers", type="primary", icon=":material/send:")

if show:
    if question not in st.session_state.asked:
        st.session_state.asked.add(question)
        winner = by_question[question]["Best Answer"]
        st.session_state.tally[winner] = st.session_state.tally.get(winner, 0) + 1
    st.session_state.last_question = question

if st.session_state.last_question:
    row = by_question[st.session_state.last_question]
    winner = row["Best Answer"]

    labels = {"Base": "Base model (untrained)", "SFT": "SFT (instruction fine-tuned)", "DPO": "DPO (final model)"}
    cols = st.columns(3, border=True)
    for col, key in zip(cols, ["Base", "SFT", "DPO"]):
        with col:
            if key == winner:
                st.markdown(f"**{labels[key]}** :green-badge[:material/trophy: Best answer]")
            else:
                st.markdown(f"**{labels[key]}** :red-badge[Not the best here]")
            st.write(row[key])

    st.subheader(f":material/lightbulb: Why {labels[winner]} wins here", divider="gray")
    st.info(row["Reason"], icon=":material/info:")


Overwriting app.py


## Launch it
Runs the Streamlit server in the background, then opens it in a new browser
tab using Colab's own port-forwarding -- no ngrok account or token needed.

In [141]:
import subprocess, time, urllib.request

# kill any previous instance still running from an earlier cell run
subprocess.run(['pkill', '-f', 'streamlit run'], capture_output=True)
time.sleep(1)

log_file = open('streamlit.log', 'w')
proc = subprocess.Popen(
    [
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        # required for the app to actually render (not just load a blank shell)
        # when accessed through Colab's proxied iframe instead of a normal browser tab
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ],
    stdout=log_file, stderr=subprocess.STDOUT,
)

# actively wait until the server is really answering, instead of a blind sleep
for attempt in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        print(f'Streamlit is up after {attempt+1}s (pid {proc.pid}).')
        print('Do NOT click any "Local URL" / "Network URL" / "External URL" '
              'text you may see below or in streamlit.log -- those are internal '
              'VM addresses your browser cannot reach. Run the next two cells '
              '(markdown note, then serve_kernel_port_as_iframe) to actually view the app.')
        break
    except Exception:
        if proc.poll() is not None:
            print('Streamlit process exited early -- see the log below:')
            print(open('streamlit.log').read())
            break
else:
    print('Streamlit did not respond in time -- run the diagnostic cell below to see streamlit.log for errors.')

Streamlit is up after 3s (pid 28355).
Do NOT click any "Local URL" / "Network URL" / "External URL" text you may see below or in streamlit.log -- those are internal VM addresses your browser cannot reach. Run the next two cells (markdown note, then serve_kernel_port_as_iframe) to actually view the app.


**Important:** the app will render directly inside this notebook, embedded
below the next cell (via an iframe) — that's the only working link.

If you scroll up into the launch cell's output/log and see lines like
`Local URL: http://localhost:8501`, `Network URL: ...`, or
`External URL: http://<something>:8501` — **do not click those.** They're
the raw addresses the Streamlit process binds to *inside the Colab VM*,
which your browser has no route to; clicking them just reloads/fails to
connect. They're diagnostic text, not links meant for you. Always use the
`serve_kernel_port_as_iframe` cell below instead — that's what actually
tunnels the port through Colab's proxy to your browser.

In [142]:
# diagnostic -- run this any time to check what's actually going on
print("=== is streamlit installed? ===")
!which streamlit

print("\n=== does app.py exist? ===")
!ls -la app.py

print("\n=== is anything running on port 8501? ===")
!ps aux | grep streamlit

# the log's own Local/Network/External URL lines are filtered out here on
# purpose -- they're addresses inside the Colab VM, not clickable from your
# browser, and only cause confusion. The iframe cell below is the real view.
print("\n=== streamlit.log contents (URL lines hidden -- see note above) ===")
!grep -v "URL:" streamlit.log 2>&1 || echo "(log file doesn't exist yet)"

=== is streamlit installed? ===
/usr/local/bin/streamlit

=== does app.py exist? ===
-rw-r--r-- 1 root root 6779 Jul 11 19:15 app.py

=== is anything running on port 8501? ===
root       28355 74.6  0.5 525500 71440 ?        Sl   19:15   0:02 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false
root       28402  0.0  0.0   7372  3384 ?        S    19:16   0:00 /bin/bash -c ps aux | grep streamlit
root       28404  0.0  0.0   6480  2488 ?        S    19:16   0:00 grep streamlit

=== streamlit.log contents (URL lines hidden -- see note above) ===


2026-07-11 19:16:01.599 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.




In [143]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

## Stopping the app
Run this when you're done, or just disconnect the runtime -- it'll clean up
on its own either way.

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'streamlit run'], capture_output=True)
print('Stopped.')